## CODE WITH ARI SCORE AND OUTPUT


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import timm
from torchvision import transforms
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import normalize
from scipy.spatial.distance import cdist

# --- 1. CONFIGURATION ---
BASE_PATH = "animal-clef-2026"
METADATA_PATH = os.path.join(BASE_PATH, "metadata.csv")
SAMPLE_SUB_PATH = os.path.join(BASE_PATH, "sample_submission.csv")
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. MODEL & PREPROCESSING ---
print("Loading MegaDescriptor-L...")
model_name = "hf-hub:BVRA/MegaDescriptor-L-384"
feature_extractor = timm.create_model(model_name, pretrained=True, num_classes=0)
if torch.cuda.device_count() > 1:
    feature_extractor = nn.DataParallel(feature_extractor)
feature_extractor.to(DEVICE).eval()

preprocess = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# --- 3. HELPER FUNCTIONS ---
def get_embeddings(path_list):
    dataset = WildlifeDataset(path_list, preprocess)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    all_feats = []
    with torch.no_grad():
        for batch in loader:
            feats = feature_extractor(batch.to(DEVICE))
            all_feats.append(feats.cpu().numpy())
    # CRITICAL: L2 Normalize the embeddings for Cosine Similarity
    return normalize(np.concatenate(all_feats), axis=1)

class WildlifeDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        path = os.path.join(BASE_PATH, self.paths[idx])
        img = Image.open(path).convert('RGB')
        return self.transform(img)

# --- 4. DATA LOAD ---
metadata = pd.read_csv(METADATA_PATH)
train_df = metadata[metadata['split'] == 'train'].reset_index(drop=True)
test_df = metadata[metadata['split'] == 'test'].reset_index(drop=True)

# --- 5. LOCAL VALIDATION WITH AUTO-TUNING ---
print("\n--- Tuning Local Validation ---")
val_df = train_df.sample(frac=0.1, random_state=42)
val_db = train_df.drop(val_df.index)

val_db_embs = get_embeddings(val_db['path'].tolist())
val_test_embs = get_embeddings(val_df['path'].tolist())
dists_val = cdist(val_test_embs, val_db_embs, metric='cosine')

best_ari = 0
best_threshold = 0.35

# Test various thresholds to find what hits 0.70
for thresh in [0.2, 0.3, 0.4, 0.5, 0.6]:
    val_preds = []
    for i in range(len(val_df)):
        m_idx = np.argmin(dists_val[i])
        if dists_val[i][m_idx] < thresh:
            val_preds.append(val_db.iloc[m_idx]['identity'])
        else:
            val_preds.append(f"new_{i}") # Give unique ID to each 'new' discovery for ARI calc

    score = adjusted_rand_score(val_df['identity'], val_preds)
    print(f"Threshold {thresh:.2f} -> ARI: {score:.4f}")
    if score > best_ari:
        best_ari = score
        best_threshold = thresh

print(f"\nOPTIMIZED THRESHOLD: {best_threshold} (Local ARI: {best_ari:.4f})")

# --- 6. FINAL INFERENCE USING BEST_THRESHOLD ---
print("\n--- Final Inference for Submission ---")
# Build global database
train_embs = get_embeddings(train_df['path'].tolist())
final_results = []

for species in metadata['dataset'].unique():
    sp_train = train_df[train_df['dataset'] == species]
    sp_test = test_df[test_df['dataset'] == species]
    if sp_test.empty: continue

    test_embs = get_embeddings(sp_test['path'].tolist())
    preds = []

    if not sp_train.empty:
        sp_train_embs = train_embs[sp_train.index.values]
        dists = cdist(test_embs, sp_train_embs, metric='cosine')
        for i in range(len(sp_test)):
            m_idx = np.argmin(dists[i])
            if dists[i][m_idx] < best_threshold:
                preds.append(f"cluster_{sp_train.iloc[m_idx]['identity']}")
            else:
                preds.append("DISCOVER")
    else:
        preds = ["DISCOVER"] * len(sp_test)

    # Use DBSCAN for discovery grouping
    new_mask = np.array([p == "DISCOVER" for p in preds])
    if any(new_mask):
        discovery_feats = test_embs[new_mask]
        clusterer = DBSCAN(eps=best_threshold, min_samples=1, metric='cosine').fit(discovery_feats)
        labels = clusterer.labels_
        ptr = 0
        for j in range(len(preds)):
            if preds[j] == "DISCOVER":
                preds[j] = f"cluster_{species}_new_{labels[ptr]}"
                ptr += 1

    for img_id, cluster in zip(sp_test['image_id'], preds):
        final_results.append({"image_id": img_id, "cluster": cluster})

# Save
submission = pd.DataFrame(final_results)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
final_sub = sample_sub[['image_id']].merge(submission, on='image_id', how='left').fillna("unknown")
final_sub.to_csv("submission.csv", index=False)
print("Submission saved using best threshold!")

## VISUALIZATION

In [ ]:
# FULL VISUALIZATION PIPELINE


import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import timm
from torchvision import transforms
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import normalize
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import seaborn as sns

# --- CONFIG ---
BASE_PATH = "animal-clef-2026"
METADATA_PATH = os.path.join(BASE_PATH, "metadata.csv")
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- MODEL ---
print("Loading model...")
model_name = "hf-hub:BVRA/MegaDescriptor-L-384"
model = timm.create_model(model_name, pretrained=True, num_classes=0)
model.to(DEVICE).eval()

# --- PREPROCESS ---
transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# --- DATASET ---
class WildlifeDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(os.path.join(BASE_PATH, self.paths[idx])).convert("RGB")
        return transform(img)

# --- EMBEDDING FUNCTION ---
def get_embeddings(paths):
    loader = DataLoader(WildlifeDataset(paths), batch_size=BATCH_SIZE, shuffle=False)
    feats = []
    with torch.no_grad():
        for batch in loader:
            out = model(batch.to(DEVICE))
            feats.append(out.cpu().numpy())
    return normalize(np.vstack(feats), axis=1)

# --- LOAD DATA ---
metadata = pd.read_csv(METADATA_PATH)
train_df = metadata[metadata['split'] == 'train'].reset_index(drop=True)

# --- SAMPLE FOR VISUALIZATION (reduce size if needed) ---
sample_df = train_df.sample(n=500, random_state=42)

print("Extracting embeddings...")
embeddings = get_embeddings(sample_df['path'].tolist())

labels = pd.factorize(sample_df['identity'])[0]

# 1. t-SNE VISUALIZATION

print("Running t-SNE...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_emb = tsne.fit_transform(embeddings)

plt.figure(figsize=(8,6))
plt.scatter(tsne_emb[:,0], tsne_emb[:,1], c=labels, cmap='tab20', s=10)
plt.title("t-SNE Visualization of Embeddings")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.colorbar()
plt.show()

# 2. PCA VISUALIZATION

print("Running PCA...")
pca = PCA(n_components=2)
pca_emb = pca.fit_transform(embeddings)

plt.figure(figsize=(8,6))
plt.scatter(pca_emb[:,0], pca_emb[:,1], c=labels, cmap='tab20', s=10)
plt.title("PCA Visualization of Embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar()
plt.show()

# 3. DISTANCE DISTRIBUTION

print("Plotting distance distribution...")
dists = cdist(embeddings, embeddings, metric='cosine')

plt.figure(figsize=(8,5))
sns.histplot(dists.flatten(), bins=50, kde=True)
plt.title("Cosine Distance Distribution")
plt.xlabel("Distance")
plt.ylabel("Frequency")
plt.show()

# 4. THRESHOLD VS ARI

print("Evaluating thresholds...")
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6]
scores = []

for thresh in thresholds:
    preds = []
    for i in range(len(sample_df)):
        d = dists[i]
        idx = np.argmin(d)
        if d[idx] < thresh:
            preds.append(sample_df.iloc[idx]['identity'])
        else:
            preds.append(f"new_{i}")
    ari = adjusted_rand_score(sample_df['identity'], preds)
    scores.append(ari)

plt.figure(figsize=(8,5))
plt.plot(thresholds, scores, marker='o')
plt.title("Threshold vs ARI")
plt.xlabel("Threshold")
plt.ylabel("ARI Score")
plt.grid()
plt.show()


# 5. DBSCAN CLUSTER VISUALIZATION

print("Running DBSCAN...")
clusterer = DBSCAN(eps=0.4, min_samples=2, metric='cosine').fit(embeddings)
cluster_labels = clusterer.labels_

pca_emb = PCA(n_components=2).fit_transform(embeddings)

plt.figure(figsize=(8,6))
plt.scatter(pca_emb[:,0], pca_emb[:,1], c=cluster_labels, cmap='tab20', s=10)
plt.title("DBSCAN Cluster Visualization")
plt.xlabel("X")
plt.ylabel("Y")
plt.colorbar()
plt.show()

# 6. SAMPLE CLUSTER IMAGES
print("Showing sample clusters...")

import random

unique_clusters = list(set(cluster_labels))
chosen = random.sample(unique_clusters, min(3, len(unique_clusters)))

for cl in chosen:
    print(f"\nCluster {cl}")
    idxs = [i for i, c in enumerate(cluster_labels) if c == cl][:5]

    plt.figure(figsize=(10,2))
    for i, idx in enumerate(idxs):
        img = Image.open(os.path.join(BASE_PATH, sample_df.iloc[idx]['path']))
        plt.subplot(1, len(idxs), i+1)
        plt.imshow(img)
        plt.axis('off')
    plt.show()

print("Visualization complete!")